In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import numpy as np
import xgboost as xgb
from helper import *
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------
set_seed(42)
merged_building_df = pd.read_csv('/home/theo/Desktop/CESI_DUTh/XAI_Energy_Consumption_Prediction/Data/merged_sud_with_power.csv')

metric_datasets = extract_metric_datasets(merged_building_df)
co2_df = metric_datasets.get('CO2 indoor')
humidity_df = metric_datasets.get('HUM indoor')
temp_df = metric_datasets.get('TMP indoor')

df = merged_building_df

🔑 All random seeds set to: 42


In [2]:
# ---------------------------------------------------------
# 2. Clean & scale
# ---------------------------------------------------------
cols_to_drop = ['Date and Time', 'Horodatage_Début', 'Horodatage_Fin', 'Valeur']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df_numeric = df.drop(columns=cols_to_drop)
df_numeric = df_numeric.ffill().bfill()

target_col_name = 'Consommation'
assert target_col_name in df_numeric.columns, f"'{target_col_name}' not found in dataframe columns!"
target_idx = df_numeric.columns.get_loc(target_col_name)

scaler = StandardScaler()
data_scaled = scaler.fit_transform(df_numeric.values)
print(f"Transformed sequence shape: {data_scaled.shape}")

def inverse_transform_target(values_scaled, scaler, col_idx):
    mean = scaler.mean_[col_idx]
    scale = scaler.scale_[col_idx]
    return values_scaled * scale + mean

Transformed sequence shape: (9648, 101)


In [3]:
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------
# 3. Build sliding windows: ALL columns included, including
#    every lag of Consommation itself (t-0 through t-11).
# ---------------------------------------------------------
seq_length = 12
gap = seq_length

n_windows = len(data_scaled) - seq_length
split_point = int(n_windows * 0.8)

train_range = range(0, split_point - gap)
test_range = range(split_point, n_windows)

X_train, y_train = [], []
for i in train_range:
    window = data_scaled[i : i + seq_length, :]
    target = data_scaled[i + seq_length, target_idx]
    X_train.append(window.flatten())
    y_train.append(target)

X_test, y_test = [], []
for i in test_range:
    window = data_scaled[i : i + seq_length, :]
    target = data_scaled[i + seq_length, target_idx]
    X_test.append(window.flatten())
    y_test.append(target)

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Feature names: ALL columns (including Consommation), expanded across timesteps
base_feature_names = df_numeric.columns.tolist()
flat_feature_names = [
    f"{feat}_t-{seq_length-1-t}"
    for t in range(seq_length)
    for feat in base_feature_names
]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"len(flat_feature_names): {len(flat_feature_names)}")

/home/theo/miniconda3/envs/energy/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


X_train: (7696, 1212), X_test: (1928, 1212)
len(flat_feature_names): 1212


In [4]:
# # ---------------------------------------------------------
# # 5. Train XGBoost
# # ---------------------------------------------------------
# model = xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05, random_state=42)
# model.fit(X_train, y_train)

# # ---------------------------------------------------------
# # 5. Train Gradient Boosting Regressor
# # ---------------------------------------------------------
# from sklearn.ensemble import GradientBoostingRegressor
# model = GradientBoostingRegressor(
#     n_estimators=500,
#     max_depth=6,
#     learning_rate=0.05,
#     random_state=42,
# )
# model.fit(X_train, y_train)


# ---------------------------------------------------------
# 5. Train LightGBM Regressor
# ---------------------------------------------------------

# import lightgbm as lgb

# model = lgb.LGBMRegressor(
#     n_estimators=500,
#     max_depth=7,
#     learning_rate=0.05,
#     random_state=42,
#     verbose=-1,   # suppress LightGBM's default per-iteration logging
# )
# model.fit(X_train, y_train)

# ---------------------------------------------------------
# 5. Train Linear Regression
# ---------------------------------------------------------
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1212,)","[-0.07,-0.18,-0.01,..., 0. , 0. , 1.04]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.004426
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1212
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,1212
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](1212,)","[1658.7 ,1065.38, 832.12,..., 1.1 , 1.1 , 1.09]"


In [5]:

# ---------------------------------------------------------
# 6. Evaluate
# ---------------------------------------------------------
y_pred = model.predict(X_test)
y_pred_orig = inverse_transform_target(y_pred, scaler, target_idx)
y_test_orig = inverse_transform_target(y_test, scaler, target_idx)

mae = mean_absolute_error(y_test_orig, y_pred_orig)
mse = mean_squared_error(y_test_orig, y_pred_orig)
rmse = np.sqrt(mse)
value_range = y_test_orig.max() - y_test_orig.min()
nrmse = rmse / value_range if value_range != 0 else float('nan')
r2 = r2_score(y_test_orig, y_pred_orig)

print("XGBoost Test Set Performance (original units, ALL Consommation lags included)")
print("-" * 40)
print(f"MAE   : {mae:.4f}")
print(f"MSE   : {mse:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"NRMSE : {nrmse:.4f}")
print(f"R2    : {r2:.4f}")

XGBoost Test Set Performance (original units, ALL Consommation lags included)
----------------------------------------
MAE   : 0.2964
MSE   : 0.2177
RMSE  : 0.4665
NRMSE : 0.0615
R2    : 0.9428


In [6]:

# ---------------------------------------------------------
# 7. SHAP Feature Importance
# ---------------------------------------------------------
explainer = shap.Explainer(model)
shap_values = explainer.shap_values(X_test)

print(f"shap_values shape: {shap_values.shape}")   # sanity check

mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = (
    pd.DataFrame({"feature": flat_feature_names, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
print(importance_df.head(20).to_string(index=False))

TypeError: The passed model is not callable and cannot be analyzed directly with the given masker! Model: LinearRegression()

In [ ]:

shap.summary_plot(shap_values, features=X_test, feature_names=flat_feature_names, max_display=20, show=True)
shap.summary_plot(shap_values, features=X_test, feature_names=flat_feature_names, plot_type="bar", max_display=20, show=True)

/tmp/ipykernel_46470/1387059583.py:1: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, features=X_test, feature_names=flat_feature_names, max_display=20, show=True)


<Figure size 800x950 with 2 Axes>

/tmp/ipykernel_46470/1387059583.py:2: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, features=X_test, feature_names=flat_feature_names, plot_type="bar", max_display=20, show=True)


<Figure size 800x950 with 1 Axes>